# 02 — Creative Performance Analysis

Goal: understand how KPIs are distributed and which categorical dimensions (vertical, format, theme, hook, language) separate winners from losers.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
DATA = Path("../")

cs = pd.read_csv(DATA / "creative_summary.csv")
print(f"creative_summary shape: {cs.shape}")

## 1. Creative Status Distribution

In [ ]:
status_order = ["top_performer", "stable", "fatigued", "underperformer"]
status_colors = {
    "top_performer": "#2ecc71",
    "stable": "#3498db",
    "fatigued": "#e67e22",
    "underperformer": "#e74c3c",
}

counts = cs["creative_status"].value_counts().reindex(status_order)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Bar
bars = ax1.bar(
    counts.index, counts.values, color=[status_colors[s] for s in counts.index], edgecolor="white"
)
ax1.set_title("Creative Status Counts", fontweight="bold")
ax1.set_ylabel("Number of Creatives")
for bar in bars:
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        str(int(bar.get_height())),
        ha="center",
        va="bottom",
        fontweight="bold",
    )

# Pie
ax2.pie(
    counts.values,
    labels=counts.index,
    autopct="%1.1f%%",
    colors=[status_colors[s] for s in counts.index],
    startangle=140,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
)
ax2.set_title("Creative Status Share", fontweight="bold")

plt.suptitle("Creative Status Distribution (N=1,080)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. KPI Distributions

In [ ]:
kpis = ["overall_ctr", "overall_cvr", "overall_roas", "overall_ipm"]
kpi_labels = ["CTR", "CVR", "ROAS", "IPM (Installs/1K Impr)"]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, (kpi, label) in enumerate(zip(kpis, kpi_labels)):
    for status in status_order:
        subset = cs[cs["creative_status"] == status][kpi].dropna()
        axes[i].hist(
            subset,
            bins=40,
            alpha=0.55,
            label=status,
            color=status_colors[status],
            edgecolor="none",
        )
    axes[i].set_title(f"{label} Distribution by Status", fontweight="bold")
    axes[i].set_xlabel(label)
    axes[i].legend(fontsize=8)

plt.suptitle("KPI Distributions by Creative Status", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Performance by Vertical

In [ ]:
vertical_perf = (
    cs.groupby("vertical")[["overall_ctr", "overall_cvr", "overall_roas", "perf_score"]]
    .median()
    .sort_values("perf_score", ascending=False)
)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
metrics = [
    ("overall_ctr", "Median CTR"),
    ("overall_cvr", "Median CVR"),
    ("overall_roas", "Median ROAS"),
    ("perf_score", "Median Perf Score"),
]

for ax, (col, label) in zip(axes, metrics):
    sorted_vals = vertical_perf[col].sort_values(ascending=True)
    bars = ax.barh(sorted_vals.index, sorted_vals.values, color="#4C72B0", edgecolor="white")
    ax.set_title(f"{label} by Vertical", fontweight="bold")
    ax.set_xlabel(label)

plt.suptitle("Median KPIs by Business Vertical", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nVertical ranking by perf_score:")
print(vertical_perf["perf_score"].sort_values(ascending=False).round(3))

## 4. Performance by Format

In [ ]:
fmt_perf = cs.groupby("format")[
    ["overall_ctr", "overall_cvr", "overall_roas", "perf_score"]
].median()

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(fmt_perf))
width = 0.2
palette = sns.color_palette("muted", 4)

for i, (col, label) in enumerate(
    [
        ("overall_ctr", "CTR"),
        ("overall_cvr", "CVR"),
        ("overall_roas", "ROAS"),
        ("perf_score", "Perf Score"),
    ]
):
    ax.bar(x + i * width, fmt_perf[col], width, label=label, color=palette[i], edgecolor="white")

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(fmt_perf.index, rotation=20)
ax.legend()
ax.set_title("Median Performance by Ad Format", fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Performance by Theme & Hook Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, col, title in [
    (axes[0], "theme", "Median Perf Score by Theme"),
    (axes[1], "hook_type", "Median Perf Score by Hook Type"),
]:
    data = cs.groupby(col)["perf_score"].median().sort_values(ascending=True)
    colors = ["#e74c3c" if v < data.median() else "#2ecc71" for v in data.values]
    ax.barh(data.index, data.values, color=colors, edgecolor="white")
    ax.axvline(data.median(), color="gray", linestyle="--", linewidth=1, label="median")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Median Perf Score")
    ax.legend(fontsize=8)

plt.suptitle("Creative Content Attributes vs Performance", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Performance by Dominant Color & Emotional Tone

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, col, title in [
    (axes[0], "dominant_color", "Median Perf Score by Dominant Color"),
    (axes[1], "emotional_tone", "Median Perf Score by Emotional Tone"),
]:
    data = cs.groupby(col)["perf_score"].median().sort_values(ascending=True)
    ax.barh(data.index, data.values, color="#4C72B0", edgecolor="white")
    ax.axvline(data.median(), color="gray", linestyle="--", linewidth=1)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Median Perf Score")

plt.tight_layout()
plt.show()

## 7. Spend Concentration (Pareto)

In [ ]:
spend_sorted = cs["total_spend_usd"].sort_values(ascending=False).reset_index(drop=True)
cumulative = spend_sorted.cumsum() / spend_sorted.sum() * 100
pct_creatives = (np.arange(1, len(spend_sorted) + 1) / len(spend_sorted)) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(pct_creatives, cumulative, color="steelblue", linewidth=2)
ax.axhline(80, color="red", linestyle="--", linewidth=1, label="80% of spend")
ax.axhline(50, color="orange", linestyle="--", linewidth=1, label="50% of spend")

# Find the x at which cumulative crosses 80%
idx_80 = np.searchsorted(cumulative.values, 80)
ax.axvline(pct_creatives[idx_80], color="red", linestyle=":", linewidth=1)
ax.text(
    pct_creatives[idx_80] + 1,
    20,
    f"{pct_creatives[idx_80]:.1f}% of\ncreatives",
    color="red",
    fontsize=9,
)

ax.set_xlabel("Cumulative % of Creatives (ranked by spend)")
ax.set_ylabel("Cumulative % of Total Spend")
ax.set_title("Spend Concentration (Pareto Curve)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

print(
    f"Top 10% of creatives account for {cumulative.iloc[int(len(cumulative) * 0.1)]:.1f}% of total spend"
)

## 8. Spend vs ROAS Scatter (colored by status)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

for status in status_order:
    sub = cs[cs["creative_status"] == status]
    ax.scatter(
        sub["total_spend_usd"],
        sub["overall_roas"],
        alpha=0.5,
        s=20,
        label=status,
        color=status_colors[status],
    )

ax.set_xlabel("Total Spend USD (log scale)")
ax.set_ylabel("Overall ROAS")
ax.set_xscale("log")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="ROAS = 1.0 (break-even)")
ax.set_title("Spend vs ROAS by Creative Status", fontweight="bold")
ax.legend(markerscale=2)
plt.tight_layout()
plt.show()

## 9. perf_score Box Plot by Status

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
data_by_status = [
    cs[cs["creative_status"] == s]["perf_score"].dropna().values for s in status_order
]
bp = ax.boxplot(data_by_status, labels=status_order, patch_artist=True, notch=True)
for patch, status in zip(bp["boxes"], status_order):
    patch.set_facecolor(status_colors[status])
    patch.set_alpha(0.7)
ax.set_title("perf_score Distribution by Status (validates label quality)", fontweight="bold")
ax.set_ylabel("perf_score")
ax.set_xlabel("Creative Status")
plt.tight_layout()
plt.show()

## 10. Top 20 Creatives by perf_score

In [ ]:
top20 = cs.nlargest(20, "perf_score")[
    [
        "creative_id",
        "vertical",
        "format",
        "theme",
        "hook_type",
        "dominant_color",
        "emotional_tone",
        "creative_status",
        "overall_ctr",
        "overall_roas",
        "perf_score",
    ]
].set_index("creative_id")

print("Top 20 creatives by perf_score:")
print(top20.to_string())